In [2]:
from google.colab import userdata
from huggingface_hub import login
import sys
import os

os.environ['GITHUB_TOKEN'] = userdata.get('GITHUB_TOKEN')
!git clone https://$GITHUB_TOKEN@github.com/Constantine1824/TRI-AI-SLM.git
sys.path.append('/content/TRI-AI-SLM')
login(token=userdata.get('HF_TOKEN'))
%cd /content/TRI-AI-SLM
!git pull origin main

fatal: destination path 'TRI-AI-SLM' already exists and is not an empty directory.
/content/TRI-AI-SLM
From https://github.com/Constantine1824/TRI-AI-SLM
 * branch            main       -> FETCH_HEAD
Already up to date.


In [3]:
import importlib
importlib.invalidate_caches()

import importlib.util
print(importlib.util.find_spec('finetune'))

ModuleSpec(name='finetune', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7d43c28338f0>, origin='/content/TRI-AI-SLM/finetune/__init__.py', submodule_search_locations=['/content/TRI-AI-SLM/finetune'])


In [ ]:
!pip install -r requirements.txt
import numpy as np
import pandas as pd
from finetune.trainer import collate_fn, finetune, evaluate
from rag.embedding import load_embedder
from rag.retriever import HybridRetriever
from utils.format import format_train_data, format_test_data

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 13.9 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.47k [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

In [5]:
data = pd.read_csv('data/train_qa.csv')
data.head()

,question,topic,care_setting,population,document_id,reference_answer,QuestionId
0,BP readings improved — skip tablets this weekend?,chronic_disease,primary_care,adult,doc_chr_002,No — never stop antihypertensives abruptly wit...,1
1,Pregnant and always tired — common workup?,maternal_health,primary_care,pregnant,doc_mat_003,Screen for anaemia and treat with iron plus di...,2
2,When should someone with flu-like symptoms sta...,infectious_disease,community,general,doc_inf_002,While febrile to reduce spread at school or work.,3
3,How do I manage type 2 diabetes day to day?,chronic_disease,primary_care,adult,doc_chr_001,"Balance meals, stay active, take medicines as ...",4
4,Feverish toddler — paracetamol cautions?,medication_safety,home,child,doc_med_003,Use weight-based doses and avoid exceeding dai...,5


In [6]:
doc = pd.read_csv('data/documents.csv')
doc.head()

,document_id,title,topic,care_setting,population,text,origin,source_url,license
0,doc_chr_001,Type 2 diabetes self-management,chronic_disease,primary_care,adult,Type 2 diabetes management combines balanced m...,synthetic,NaN,CC0-1.0
1,doc_chr_002,Hypertension lifestyle measures,chronic_disease,primary_care,adult,"Lowering dietary salt, maintaining healthy wei...",synthetic,NaN,CC0-1.0
2,doc_chr_003,Asthma action plan basics,chronic_disease,primary_care,child,Children with asthma should use a written acti...,synthetic,NaN,CC0-1.0
3,doc_inf_001,Malaria prevention in endemic areas,infectious_disease,community,general,"Sleep under insecticide-treated nets, eliminat...",synthetic,NaN,CC0-1.0
4,doc_inf_002,Hand hygiene and respiratory etiquette,infectious_disease,community,general,Wash hands with soap for at least twenty secon...,synthetic,NaN,CC0-1.0


In [7]:
data = data.merge(doc[['document_id', 'text']], on='document_id', how='left')
data = data.rename(columns={'text':'context'})
data.head()

,question,topic,care_setting,population,document_id,reference_answer,QuestionId,context
0,BP readings improved — skip tablets this weekend?,chronic_disease,primary_care,adult,doc_chr_002,No — never stop antihypertensives abruptly wit...,1,"Lowering dietary salt, maintaining healthy wei..."
1,Pregnant and always tired — common workup?,maternal_health,primary_care,pregnant,doc_mat_003,Screen for anaemia and treat with iron plus di...,2,Anaemia increases fatigue and adverse birth ou...
2,When should someone with flu-like symptoms sta...,infectious_disease,community,general,doc_inf_002,While febrile to reduce spread at school or work.,3,Wash hands with soap for at least twenty secon...
3,How do I manage type 2 diabetes day to day?,chronic_disease,primary_care,adult,doc_chr_001,"Balance meals, stay active, take medicines as ...",4,Type 2 diabetes management combines balanced m...
4,Feverish toddler — paracetamol cautions?,medication_safety,home,child,doc_med_003,Use weight-based doses and avoid exceeding dai...,5,Use weight-based paediatric paracetamol dosing...


In [ ]:
from datasets import Dataset

test_data = pd.read_csv('data/test_questions.csv')
retriever = HybridRetriever(embedder=load_embedder('tfidf'))

data['context'] = [
    retriever.context_for_document(document_id)
    for document_id in data['document_id']
]
test_data['context'] = [
    retriever.retrieve_context(
        row['question'], row['topic'], row['care_setting'], row['population'], k=3
    )
    for _, row in test_data.iterrows()
]

train_data = Dataset.from_pandas(data, preserve_index=False)
test_data = Dataset.from_pandas(test_data, preserve_index=False)
train_data = train_data.map(format_train_data)
test_data = test_data.map(format_test_data)

Map:   0%|          | 0/43 [00:00<?, ? examples/s]

Map:   0%|          | 0/11 [00:00<?, ? examples/s]

In [9]:
trainer = finetune(train_data)
eval = evaluate(trainer, test_data, batch_size=11)
eval

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


Step,Training Loss
4,3.350792
8,2.201967
12,1.285489
16,0.733576
20,0.496200
24,0.378125
28,0.297836
32,0.214249
36,0.160476
40,0.107477


['\nAnswer:Iron supplementation, deworming, and dietary counselling for anaemia.',
 '\nAnswer:Truekelp for fever under 3 months unless clinician advises otherwise. Watch for lethargy, poor feeding, or stiff neck.',
 '\nAnswer:Severe bleeding, abdominal pain, reduced fetal movement, or active preterm labour.',
 '\nAnswer:Listen empathetically, validate feelings, and recommend school counselling or helpline referral.',
 '\nAnswer:Use insecticide-treated bed nets, indoor residual spraying where recommended, and take chemoprophylaxis as directed.',
 '\nAnswer:Use a written action plan, use a reliever inhaler daily, and avoid known triggers.',
 '\nAnswer:Half plate fruits/vegetables, one quarter protein, one quarter whole grains.',
 '\nAnswer:Complete full course even if better; early stopping increases resistant bacteria.',
 '\nAnswer:Every ten years, or after a dirty wound with a minor cut risk.',
 '\nAnswer:Check for ketones in urine, treat persistent vomiting/confusion, and follow misse

In [10]:
test_df = pd.read_csv('data/test_questions.csv')
test_df.head()

,QuestionId,question,topic,care_setting,population
0,1001,How is pregnancy anaemia managed?,maternal_health,primary_care,pregnant
1,1002,Fever in a two-month-old — what to do?,emergency_triage,hospital,infant
2,1003,Pregnancy symptoms needing urgent review?,maternal_health,primary_care,pregnant
3,1004,Teen refuses school citing panic — approach?,mental_health_basics,school,child
4,1005,How can families prevent malaria?,infectious_disease,community,general


In [11]:
qid = test_df['QuestionId']
qid

,QuestionId
0,1001
1,1002
2,1003
3,1004
4,1005
5,1006
6,1007
7,1008
8,1009
9,1010


In [ ]:
def save_submission(result):
    qid = test_df['QuestionId']
    cleaned = pd.Series([
        s.lstrip('\n').removeprefix('Answer:').strip()
        for s in result
    ])
    submission = qid.to_frame()
    submission['Answer'] = cleaned
    assert list(submission.columns) == ['QuestionId', 'Answer']
    assert len(submission) == len(test_df) == 11
    assert submission['QuestionId'].tolist() == test_df['QuestionId'].tolist()
    assert submission['Answer'].str.strip().ne('').all()
    submission.to_csv('/kaggle/working/submission.csv', index=False)
    print(submission)
    print('Saved /kaggle/working/submission.csv')

save_submission(eval)

0     Iron supplementation, deworming, and dietary c...
1     Truekelp for fever under 3 months unless clini...
2     Severe bleeding, abdominal pain, reduced fetal...
3     Listen empathetically, validate feelings, and ...
4     Use insecticide-treated bed nets, indoor resid...
5     Use a written action plan, use a reliever inha...
6     Half plate fruits/vegetables, one quarter prot...
7     Complete full course even if better; early sto...
8     Every ten years, or after a dirty wound with a...
9     Check for ketones in urine, treat persistent v...
10    Blunt the stool, feed oral rehydration solutio...
dtype: object
    QuestionId                                             Answer
0         1001  Iron supplementation, deworming, and dietary c...
1         1002  Truekelp for fever under 3 months unless clini...
2         1003  Severe bleeding, abdominal pain, reduced fetal...
3         1004  Listen empathetically, validate feelings, and ...
4         1005  Use insecticide-treated 